# Time-Walk Calibration Iteration 2 (timewalk_tw) & Stability Analysis

Comprehensive examination of the **second iteration of Time-Walk Calibration** () produced after effective scintillator velocity ({\mathrm{scint}}^{\mathrm{TW}}$) compensation.

### Analysis Goals:
1. **Ingestion & Stability Tracking:** Load all calibrated runs from Category II (2022), Category III (2023), and Category III (2025).
2. **Parameter Stability Over Time:** Track fine alignment $\Delta t$ (), time-walk curvature $lpha$, scale $eta$, threshold $, and systematic error floor $\sigma_{\mathrm{sys}}$ as a function of run number.
3. **Iteration 1 vs. Iteration 2 Comparison:** Directly compare parameter convergence and residual timing resolution between  (Iteration 1) and  (Iteration 2).


In [1]:
# Fix JupyROOT post-execute hook crash (ROOT::Experimental::RCanvas::GetCanvases ValueError)
try:
    import ROOT
    try:
        import ROOT._jupyroot.helpers.utils as _jutils
        _jutils.GetRCanvasDrawers = lambda: []
        _jutils.RCanvasAvailable = lambda: False
    except (ImportError, AttributeError):
        pass
except ImportError:
    pass

try:
    from IPython import get_ipython
    _ip = get_ipython()
    if _ip and hasattr(_ip, "events"):
        for _cb in list(_ip.events.callbacks.get("post_execute", [])):
            if "CaptureDrawnPrimitives" in str(_cb) or "NotebookDraw" in str(_cb):
                _ip.events.unregister("post_execute", _cb)
except Exception:
    pass

import os
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy import stats
import concurrent.futures

try:
    import ddfUtils.mpl_styling as dms
    has_dms = True
except ImportError:
    has_dms = False

plt.rcParams.update({
    "font.size": 13,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "figure.titlesize": 15,
    "figure.autolayout": True
})
print("Environment initialized successfully.")


Environment initialized successfully.


---
## 1. Data Ingestion & Loading Utilities

Loads all calibration files into tabular datasets (`df_channels`, `df_runs`, `cat_al`), and defines `load_channel_tw_root(...)` to ingest 2D time-walk distributions, fit parameters, and residual projections directly from ROOT files.


In [ ]:
fits_dir_iter1 = "/eos/user/i/idioniso/Timing/timewalk/fits"
fits_dir_iter2 = "/eos/user/i/idioniso/Timing/timewalk_tw/fits"
runs_dir = "/eos/user/i/idioniso/Timing/timewalk_tw/runs"

def get_run_period(run: int) -> str:
    """Classifies run into data-taking periods."""
    if 5000 <= run < 6000:
        return "Cat II (2022)"
    elif 6000 <= run < 7000:
        return "Cat III (2023)"
    elif 8000 <= run < 10000:
        return "Cat III (2024)"
    elif 10000 <= run < 13000:
        return "Cat III (2025)"
    elif 13000 <= run < 14000:
        return "Cat III (2026)"
    return "Other"

def load_tw_dataset(fits_dir_iter2, tag="iter2"):
    json_files = sorted(glob.glob(os.path.join(fits_dir_iter2, "run*_timing_calibration.json")))
    
    def parse_single_calib(jf):
        with open(jf, "r") as f:
            data = json.load(f)
        r = data.get("run", 0)
        cat = data.get("category", "catII")
        period = get_run_period(r)
        al_all = data.get("alignment", {})
        tw_all = data.get("timewalk", {})

        ch_records = []
        run_records = []

        for ref in ["DS", "SF"]:
            al_dict = al_all.get(ref, {})
            tw_dict = tw_all.get(ref, {})
            if not al_dict:
                continue

            d_vals, t0_vals, a_vals, b_vals = [], [], [], []
            q_vals, g_vals, s_vals, c_vals, cs_vals = [], [], [], [], []

            for ch_key, al_info in al_dict.items():
                det_id, sipm = [int(p) for p in ch_key.split("_")]
                subsys = det_id // 10000
                plane = (det_id % 10000) // 1000
                bar = det_id % 1000
                side = "left" if sipm < 8 else "right"
                system_name = "Veto" if subsys == 1 else ("US" if subsys == 2 else "DS")
                plane_label = f"{system_name} Plane {plane + 1}"

                d_val = al_info[0]
                d_err = al_info[1] if len(al_info) > 1 else 0.0
                tw_p = tw_dict.get(ch_key, {})

                d_vals.append(d_val)
                if "t0" in tw_p and tw_p["t0"] is not None: t0_vals.append(tw_p["t0"])
                if "alpha" in tw_p and tw_p["alpha"] is not None: a_vals.append(tw_p["alpha"])
                if "beta" in tw_p and tw_p["beta"] is not None: b_vals.append(tw_p["beta"])
                if "qdc0" in tw_p and tw_p["qdc0"] is not None: q_vals.append(tw_p["qdc0"])
                if "gamma" in tw_p and tw_p["gamma"] is not None: g_vals.append(tw_p["gamma"])
                if "sigma_sys" in tw_p and tw_p["sigma_sys"] is not None: s_vals.append(tw_p["sigma_sys"])
                if "chi2_tot_ndf" in tw_p and tw_p["chi2_tot_ndf"] is not None: c_vals.append(tw_p["chi2_tot_ndf"])
                if "chi2_stat_ndf" in tw_p and tw_p["chi2_stat_ndf"] is not None: cs_vals.append(tw_p["chi2_stat_ndf"])

                ch_records.append({
                    "run": r,
                    "category": cat,
                    "period": period,
                    "ref": ref,
                    "channel": ch_key,
                    "detID": det_id,
                    "subsys": subsys,
                    "system": system_name,
                    "plane": plane,
                    "plane_label": plane_label,
                    "bar": bar,
                    "sipm": sipm,
                    "side": side,
                    "d_sipm": d_val,
                    "d_sipm_err": d_err,
                    "t0": tw_p.get("t0", np.nan),
                    "alpha": tw_p.get("alpha", np.nan),
                    "beta": tw_p.get("beta", np.nan),
                    "qdc0": tw_p.get("qdc0", np.nan),
                    "gamma": tw_p.get("gamma", np.nan),
                    "sigma_sys": tw_p.get("sigma_sys", np.nan),
                    "chi2_tot_ndf": tw_p.get("chi2_tot_ndf", np.nan),
                    "chi2_stat_ndf": tw_p.get("chi2_stat_ndf", np.nan),
                    "rms_residual": tw_p.get("rms_residual", np.nan),
                    "tag": tag,
                })

            def calc_stat(vals):
                if not vals:
                    return np.nan, np.nan, np.nan
                arr = np.array(vals, dtype=float)
                arr = arr[np.isfinite(arr)]
                if len(arr) == 0:
                    return np.nan, np.nan, np.nan
                med = float(np.median(arr))
                iqr = float(np.percentile(arr, 75) - np.percentile(arr, 25))
                sem = float(1.253 * (iqr / 1.349) / np.sqrt(len(arr))) if len(arr) > 1 else 0.0
                disp = float(iqr / 1.349)
                return med, sem, disp

            d_med, d_sem, d_disp = calc_stat(d_vals)
            t0_med, t0_sem, t0_disp = calc_stat(t0_vals)
            a_med, a_sem, a_disp = calc_stat(a_vals)
            b_med, b_sem, b_disp = calc_stat(b_vals)
            q_med, q_sem, q_disp = calc_stat(q_vals)
            g_med, g_sem, g_disp = calc_stat(g_vals)
            s_med, s_sem, s_disp = calc_stat(s_vals)
            c_med, c_sem, c_disp = calc_stat(c_vals)
            cs_med, cs_sem, cs_disp = calc_stat(cs_vals)

            run_records.append({
                "run": r,
                "category": cat,
                "period": period,
                "ref": ref,
                "tag": tag,
                "n_channels": len(d_vals),
                "d_median": d_med, "d_sem": d_sem,
                "t0_median": t0_med, "t0_sem": t0_sem,
                "alpha_median": a_med, "alpha_sem": a_sem,
                "beta_median": b_med, "beta_sem": b_sem,
                "qdc0_median": q_med, "qdc0_sem": q_sem,
                "gamma_median": g_med, "gamma_sem": g_sem,
                "sigma_sys_median": s_med, "sigma_sys_sem": s_sem,
                "chi2_tot_median": c_med, "chi2_tot_sem": c_sem,
                "chi2_stat_median": cs_med, "chi2_stat_sem": cs_sem,
            })
        return ch_records, run_records

    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
        parsed_results = list(executor.map(parse_single_calib, json_files))

    channel_records = [rec for p in parsed_results for rec in p[0]]
    run_summary_records = [rec for p in parsed_results for rec in p[1]]

    return pd.DataFrame(channel_records), pd.DataFrame(run_summary_records)

print("Loading Iteration 2 (timewalk_tw)... ")
df_channels_tw, df_runs_tw = load_tw_dataset(fits_dir_iter2, tag="iter2")
print(f"  -> Loaded {len(df_runs_tw)//2} runs ({len(df_channels_tw):,} channel records)")

print("Loading Iteration 1 (timewalk)... ")
df_channels_iter1, df_runs_iter1 = load_tw_dataset(fits_dir_iter1, tag="iter1")
print(f"  -> Loaded {len(df_runs_iter1)//2} runs ({len(df_channels_iter1):,} channel records)")

# Convenience pointers for stability plotting
df_channels = df_channels_tw
df_runs = df_runs_tw


def load_channel_tw_root(sample_run: int, target_ch: str, ref_type: str = "DS",
                         y_min: float = -20.0, y_max: float = 20.0, n_y_bins: int = 200) -> dict:
    """Loads 2D dt vs QDC histogram and 1D residual distribution from the merged run ROOT file."""
    import ROOT
    ROOT.gROOT.SetBatch(True)
    
    root_file_path = os.path.join(runs_dir, f"run{sample_run}.root")
    json_file_path = os.path.join(fits_dir_iter2, f"run{sample_run}_timing_calibration.json")
    
    if not os.path.exists(json_file_path):
        raise FileNotFoundError(f"Cannot open {json_file_path}")
    with open(json_file_path, "r") as f:
        calib_data = json.load(f)
    tw_params = calib_data.get("timewalk", {}).get(ref_type, {}).get(target_ch, {})
    if not tw_params:
        raise KeyError(f"Channel {target_ch} ({ref_type}) not found in {json_file_path}")
    
    rf = ROOT.TFile.Open(root_file_path, "READ")
    if not rf or rf.IsZombie():
        raise FileNotFoundError(f"Cannot open {root_file_path}")
    
    h2 = rf.Get(f"{target_ch}/dtvqdc-{ref_type}")
    h_res = rf.Get(f"{target_ch}/dt_residual_{ref_type}_{target_ch}")
    if not h2:
        rf.Close()
        raise KeyError(f"Histogram {target_ch}/dtvqdc-{ref_type} not found in {root_file_path}")
    
    nbins_x = h2.GetNbinsX()
    nbins_y = h2.GetNbinsY()
    x_min, x_max = 0.0, 200.0
    x_edges = np.linspace(x_min, x_max, nbins_x + 1)
    y_edges = np.linspace(y_min, y_max, n_y_bins + 1)
    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    
    alpha = tw_params.get("alpha", 0.0)
    beta = tw_params.get("beta", 1.0)
    qdc0 = tw_params.get("qdc0", 0.0)
    gamma = tw_params.get("gamma", 0.0)
    tw_corr = (alpha / (beta * x_centers - qdc0)) + gamma * x_centers
    
    raw_2d = np.zeros((nbins_y, nbins_x), dtype=np.float64)
    corr_2d = np.zeros((n_y_bins, nbins_x), dtype=np.float64)
    
    for bx in range(1, nbins_x + 1):
        shift = tw_corr[bx - 1]
        for by in range(1, nbins_y + 1):
            cnt = h2.GetBinContent(bx, by)
            if cnt <= 0: continue
            raw_2d[by - 1, bx - 1] = cnt
            dt_val = h2.GetYaxis().GetBinCenter(by)
            dt_tw = dt_val - shift
            if y_min <= dt_tw <= y_max:
                by_idx = int((dt_tw - y_min) / ((y_max - y_min) / n_y_bins))
                if 0 <= by_idx < n_y_bins:
                    corr_2d[by_idx, bx - 1] += cnt
                    
    raw_y_edges = np.array([h2.GetYaxis().GetBinLowEdge(i) for i in range(1, nbins_y + 2)])
    
    res_x, res_y = [], []
    sigma_dist = h_res.GetStdDev() if h_res else np.nan
    if h_res:
        for b in range(1, h_res.GetNbinsX() + 1):
            xc = h_res.GetBinCenter(b)
            if y_min <= xc <= y_max:
                res_x.append(xc)
                res_y.append(h_res.GetBinContent(b))
    rf.Close()
    
    return {
        "run": sample_run,
        "channel": target_ch,
        "ref": ref_type,
        "params": tw_params,
        "x_edges": x_edges,
        "x_centers": x_centers,
        "y_edges": y_edges,
        "raw_y_edges": raw_y_edges,
        "raw_2d": raw_2d,
        "corr_2d": corr_2d,
        "tw_corr": tw_corr,
        "res_x": np.array(res_x),
        "res_y": np.array(res_y),
        "sigma_dist": sigma_dist,
    }

print("Data loading utility load_channel_tw_root defined successfully.")


# Compute robust empirical parameter uncertainty per SiPM channel from intra-year run scatter
def get_iqr_disp(x):
    vals = x.dropna().values
    return (np.percentile(vals, 75) - np.percentile(vals, 25)) / 1.349 if len(vals) > 3 else np.nan

ch_stat_disp = df_channels_tw[df_channels_tw["period"] == "Cat II (2022)"].groupby(["channel", "ref"])[["d_sipm", "alpha", "beta", "qdc0", "sigma_sys"]].agg(get_iqr_disp).reset_index()
ch_stat_disp.columns = ["channel", "ref", "d_err_emp", "alpha_err", "beta_err", "qdc0_err", "sigma_sys_err"]

df_channels_tw = pd.merge(df_channels_tw, ch_stat_disp, on=["channel", "ref"], how="left")
df_channels_tw["d_tot_err"] = np.sqrt(df_channels_tw["d_sipm_err"].fillna(0.001)**2 + df_channels_tw["d_err_emp"].fillna(0.035)**2)
df_channels = df_channels_tw


Loading Iteration 2 (timewalk_tw)... 


---
## 2. Parameter Stability Across Time (Run Number)

Visualizes the run-by-run evolution of all Time-Walk and Alignment parameters across the detector:
- $t_0$ [ns]: Rational offset / reference time shift
- $\alpha$: Scale parameter
- $\beta$: Curvature parameter
- $\mathrm{QDC}_0$: Charge threshold / horizontal offset
- $\gamma$: Linear asymptotic slope
- $\sigma_{\mathrm{sys}}$ [ns]: Systematic error floor
- $\chi^2_{\mathrm{tot}} / \mathrm{ndf}$: Fit quality
- $d_{\mathrm{SiPM}}$ [ns]: Fine alignment parameter
- Number of Fitted Channels ($N_{\mathrm{channels}}$)

Results are separated into dedicated subfigures for each data-taking period:
1. **Category II (2022 Runs)**: 33 runs (5119–5408)
2. **Category III (2023 Runs)**: 10 runs (6079–6667)
3. **Category III (2024 Runs)**: 10 runs (8638–9880)
4. **Category III (2025 Runs)**: 10 runs (11124–12217)
5. **Category III (2026 Runs)**: 10 runs (13085–13520)


In [ ]:
def plot_tw_parameter_stability(df_runs, period_name, target=None, show_median_line=True):
    """
    Plots the run-by-run stability of all Time-Walk and Alignment parameters for a given period.
    Parameters plotted: t0, alpha, beta, QDC0, gamma, sigma_sys, chi2_tot/ndf, d_SiPM, and channel count.
    Supports either standalone figure creation or rendering into a matplotlib SubFigure.
    """
    df_period = df_runs[df_runs["period"] == period_name].copy()
    if df_period.empty:
        print(f"No run records found for period: {period_name}")
        if target is not None:
            ax = target.subplots(1, 1)
            ax.text(0.5, 0.5, f"{period_name}\n(Pending Time-Walk Calibration)",
                    ha="center", va="center", fontsize=15, fontweight="bold",
                    color="dimgray", transform=ax.transAxes)
            ax.axis("off")
        return None

    is_standalone = (target is None)
    if is_standalone:
        fig, axes = plt.subplots(3, 3, figsize=(18, 12), sharex=True)
    else:
        axes = target.subplots(3, 3, sharex=True)

    axes_flat = axes.flatten()

    param_configs = [
        ("t0_median",        "t0_sem",        "t0 [ns]",                   "Offset Parameter (t0)",          "#1f77b4"),
        ("alpha_median",     "alpha_sem",     r"\alpha",                   "Scale Parameter (\alpha)",        "#ff7f0e"),
        ("beta_median",      "beta_sem",      r"\beta",                    "Curvature Parameter (\beta)",     "#2ca02c"),
        ("qdc0_median",      "qdc0_sem",      r"\mathrm{QDC}_0",           "Charge Offset (QDC0)",            "#d62728"),
        ("gamma_median",     "gamma_sem",     r"\gamma",                   "Linear Slope (\gamma)",          "#9467bd"),
        ("sigma_sys_median", "sigma_sys_sem", r"\sigma_{\mathrm{sys}} [ns]", "Systematic Uncertainty Floor", "#8c564b"),
        ("chi2_tot_median",  "chi2_tot_sem",  r"\chi^2_{\mathrm{tot}}/\mathrm{ndf}", "Fit Quality (\chi^2/ndf)", "#e377c2"),
        ("d_median",         "d_sem",         r"d_{\mathrm{SiPM}} [ns]",   "SiPM Alignment Shift (d)",       "#7f7f7f"),
        ("n_channels",       None,            r"N_{\mathrm{channels}}",    "Fitted Channels Count",          "#bcbd22"),
    ]

    for ax, (med_col, sem_col, ylabel, title, col) in zip(axes_flat, param_configs):
        for ref_type, marker, ls in [("DS", "o", "-"), ("SF", "s", "--")]:
            df_ref = df_period[df_period["ref"] == ref_type].sort_values("run")
            if df_ref.empty:
                continue

            runs = df_ref["run"].values
            y_vals = df_ref[med_col].values
            y_err = df_ref[sem_col].values if (sem_col and sem_col in df_ref.columns) else None

            ax.errorbar(
                runs, y_vals, yerr=y_err,
                fmt=marker, linestyle=ls, linewidth=1.2, markersize=4,
                color=col if ref_type == "DS" else "#555555",
                ecolor=col if ref_type == "DS" else "#555555",
                capsize=2, elinewidth=1.0, alpha=0.9,
                label=f"{ref_type} Reference"
            )

            if show_median_line and len(y_vals) > 0 and not np.all(np.isnan(y_vals)):
                global_med = np.nanmedian(y_vals)
                ax.axhline(global_med, color=col if ref_type == "DS" else "#555555",
                           linestyle=":", linewidth=1.0, alpha=0.6)

        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(title, fontsize=11, fontweight="bold", pad=5)
        ax.grid(True, linestyle=":", alpha=0.5)
        ax.tick_params(labelsize=9)

    for ax in axes[-1, :]:
        ax.set_xlabel("Run Number", fontsize=10)

    handles, labels = axes_flat[0].get_legend_handles_labels()
    if handles:
        axes_flat[0].legend(handles, labels, loc="upper right", fontsize=8.5, framealpha=0.9)

    title_text = f"SND@LHC — Time-Walk & Alignment Stability: {period_name}"
    if is_standalone:
        fig.suptitle(title_text, fontsize=15, fontweight="bold", y=0.995)
        plt.tight_layout()
        return fig
    else:
        target.suptitle(title_text, fontsize=13, fontweight="bold", y=0.995)
        return target


### 2.1 Category II (2022 Runs)

Plots parameter trends for **Cat II (2022)** runs.


In [ ]:
fig_catII_2022 = plot_tw_parameter_stability(df_runs, "Cat II (2022)")
plt.show()


### 2.2 Category III (2023 Runs)

Plots parameter trends for **Cat III (2023)** runs.


In [ ]:
fig_catIII_2023 = plot_tw_parameter_stability(df_runs, "Cat III (2023)")
plt.show()


### 2.3 Category III (2024 Runs)

Plots parameter trends for **Cat III (2024)** runs (Runs 8638 to 9880) across all 9 fit metrics.

In [ ]:
fig_catIII_2024 = plot_tw_parameter_stability(df_runs, "Cat III (2024)")
plt.show()

### 2.4 Category III (2025 Runs)

Plots parameter trends for **Cat III (2025)** runs.


In [ ]:
fig_catIII_2025 = plot_tw_parameter_stability(df_runs, "Cat III (2025)")
plt.show()


### 2.5 Category III (2026 Runs)

Plots parameter trends for **Cat III (2026)** runs.


In [ ]:
fig_catIII_2026 = plot_tw_parameter_stability(df_runs, "Cat III (2026)")
plt.show()


### 2.6 Combined Multi-Period Subfigures Canvas

Arranges all five periods into a single 4-quadrant layout using `matplotlib.figure.SubFigure`.


In [ ]:
fig_all = plt.figure(figsize=(26, 33))
subfigs = fig_all.subfigures(3, 2)
all_periods = ["Cat II (2022)", "Cat III (2023)", "Cat III (2024)", "Cat III (2025)", "Cat III (2026)"]
for idx, p in enumerate(all_periods):
    sf = subfigs.flatten()[idx]
    plot_tw_parameter_stability(df_runs, p, target=sf, show_median_line=True)
# Hide the unused 6th subfigure
subfigs.flatten()[5].set_visible(False)
fig_all.suptitle("SND@LHC — Time-Walk Parameter Stability Comparison Across All Data Periods", fontsize=18, fontweight="bold", y=0.998)
plt.show()


---
## 3. Individual Channel Variations Across Time

Tracks representative individual channels across different planes and subsystems vs. the global run median (similar to the individual channel inspection in `cscint.ipynb`).


In [ ]:
representative_channels = [
    ("10003_3",  "Veto Pl 1, Bar 3, Left SiPM (DS ref)",  "#9467bd", "DS"),
    ("20004_4",  "US Pl 1, Bar 4, Left SiPM (DS ref)",    "#1f77b4", "DS"),
    ("20004_12", "US Pl 1, Bar 4, Right SiPM (DS ref)",   "#ff7f0e", "DS"),
    ("22005_11", "US Pl 3, Bar 5, Right SiPM (DS ref)",   "#2ca02c", "DS"),
    ("24000_4",  "US Pl 5, Bar 0, Left SiPM (DS ref)",    "#d62728", "DS"),
]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True, gridspec_kw={"height_ratios": [2.5, 1.2]})

# Global DS median baseline
df_ds_runs = df_runs[df_runs["ref"] == "DS"].sort_values("run")
ax1.plot(df_ds_runs["run"], df_ds_runs["d_median"], color="black", linewidth=2.5, linestyle="--", label="Global DS Median")

for ch_key, label_str, ch_color, ref in representative_channels:
    df_ch = df_channels[(df_channels["channel"] == ch_key) & (df_channels["ref"] == ref)].sort_values("run")
    if df_ch.empty:
        continue
    ax1.errorbar(df_ch["run"], df_ch["d_sipm"], yerr=df_ch["d_sipm_err"], fmt="-o", color=ch_color, markersize=5, label=label_str)
    # Residual to median in bottom panel
    merged = pd.merge(df_ch[["run", "d_sipm"]], df_ds_runs[["run", "d_median"]], on="run")
    ax2.plot(merged["run"], merged["d_sipm"] - merged["d_median"], marker="o", markersize=4, color=ch_color, label=ch_key)

ax1.set_ylabel(r"$d_{\mathrm{SiPM}}$ [ns]")
ax1.set_title("Individual Channel $d_{\mathrm{SiPM}}$ Stability vs. Global Median")
ax1.grid(True, alpha=0.3)
ax1.legend(loc="upper right", frameon=True, fontsize=10)

ax2.set_xlabel("Run Number")
ax2.set_ylabel(r"$\Delta d_{\mathrm{SiPM}}$ to Median [ns]")
ax2.axhline(0, color="black", linestyle=":", linewidth=1)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## 4. Full Detector Alignment Map ($d_{\mathrm{SiPM}}$ vs. SiPM Number)
### Including Both Veto and Upstream Systems

Replicates the structure of **Thesis Figure 4.38** with:
- Consecutive SiPM numbering for **Veto** (Planes 1–2, 112 channels/plane = 224 channels) and **Upstream** (Planes 1–5, 160 channels/plane = 800 channels).
- **Green solid lines** delineating detector planes.
- **Magenta dashed lines** separating **Left** (red points) and **Right** (blue points) readout sides.
- **"G" labels** indicating the SilGel side on each plane (matching Table 3.1 & Figure 4.38).


In [ ]:
# Select representative run (e.g. 5408 or 5119)
plot_run = 5408 if 5408 in df_channels["run"].values else df_channels["run"].iloc[0]
df_run = df_channels[(df_channels["run"] == plot_run) & (df_channels["ref"] == "DS")].copy()

# Build contiguous SiPM indices matching the layout:
# 1. Veto: Plane 0 (detID 10000..10006) and Plane 1 (11000..11006)
#    7 bars * 8 channels = 56 left + 56 right = 112 channels / plane
# 2. US: Planes 0..4 (detID 20000..24009)
#    10 bars * 8 channels = 80 left + 80 right = 160 channels / plane

def compute_full_sipm_index(row):
    subsys = row["subsys"]
    plane = row["plane"]
    bar = row["bar"]
    sipm = row["sipm"]
    is_left = (sipm < 8)
    sipm_in_side = sipm if is_left else (sipm - 8)
    if subsys == 1:  # Veto: offset 0 to 223
        plane_offset = plane * 112
        side_offset = 0 if is_left else 56
        return plane_offset + side_offset + bar * 8 + sipm_in_side
    elif subsys == 2:  # US: offset 224 to 1023
        us_base = 224
        plane_offset = plane * 160
        side_offset = 0 if is_left else 80
        return us_base + plane_offset + side_offset + bar * 8 + sipm_in_side
    return np.nan

df_run["global_sipm_idx"] = df_run.apply(compute_full_sipm_index, axis=1)
df_run = df_run.dropna(subset=["global_sipm_idx"]).sort_values("global_sipm_idx")

# Define plane boundaries & gel indicators
# Veto plane 0: [0, 112), left [0, 56), right [56, 112)
# Veto plane 1: [112, 224), left [112, 168), right [168, 224)
# US planes 0..4: each 160 ch
plane_boundaries = [0, 112, 224, 384, 544, 704, 864, 1024]
side_dividers = [56, 168, 304, 464, 624, 784, 944]

# Gel positions (Table 3.1 & Figure 4.38):
# US: Pl 0: Right, Pl 1: Left, Pl 2: Right, Pl 3: Left, Pl 4: Left
gel_annotations = [
    (84, "G"),   # Veto Pl 1 Right (illustrative)
    (196, "G"),  # Veto Pl 2 Right
    (344, "G"),  # US Pl 1 Right
    (424, "G"),  # US Pl 2 Left
    (664, "G"),  # US Pl 3 Right
    (744, "G"),  # US Pl 4 Left
    (904, "G"),  # US Pl 5 Left
]

fig, ax = plt.subplots(figsize=(16, 7))

# Split Left and Right points
df_left = df_run[df_run["side"] == "left"]
df_right = df_run[df_run["side"] == "right"]

ax.errorbar(df_left["global_sipm_idx"], df_left["d_sipm"], yerr=df_left["d_sipm_err"],
            fmt="+", color="red", markersize=6, capsize=0, label="left")
ax.errorbar(df_right["global_sipm_idx"], df_right["d_sipm"], yerr=df_right["d_sipm_err"],
            fmt="+", color="blue", markersize=6, capsize=0, label="right")

# Draw boundaries
for b in plane_boundaries:
    ax.axvline(b, color="lime", linewidth=2.0, linestyle="-")
for d in side_dividers:
    ax.axvline(d, color="magenta", linewidth=1.5, linestyle="--")

# Mark Gel side
y_top = ax.get_ylim()[1] if ax.get_ylim()[1] > 14 else 14.8
for x_pos, text in gel_annotations:
    ax.text(x_pos, y_top - 0.7, text, fontweight="bold", fontsize=13, ha="center", va="center")

# Label system regions
ax.text(112, y_top + 0.8, "VETO (Planes 1–2)", fontsize=12, fontweight="bold", ha="center",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="wheat", alpha=0.5))
ax.text(624, y_top + 0.8, "UPSTREAM (Planes 1–5)", fontsize=12, fontweight="bold", ha="center",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue", alpha=0.5))

ax.set_xlim(-10, 1034)
ax.set_xlabel("SiPM number (Veto: 0–223 | Upstream: 224–1023)", fontsize=14)
ax.set_ylabel(r"DS alignment parameter, $d_{\mathrm{SiPM}}$ [ns]", fontsize=14)
ax.set_title(f"DS alignment parameter, {{\mathrm{{SiPM}}}}$ as a function of SiPM number (Run {plot_run})", fontsize=15, pad=25)
ax.grid(True, which="both", linestyle=":", alpha=0.5)
ax.legend(loc="upper right", frameon=True, framealpha=0.9, facecolor="white", edgecolor="black", fontsize=13)

plt.tight_layout()
plt.show()


---
## 5. Standalone Upstream Alignment Map (Exact Replica of Thesis Figure 4.38)

Isolated view of Upstream SiPMs (indices 0 to 799) with exact 160-channel plane groupings, gel indicators, and side dividers.


In [ ]:
df_us = df_run[df_run["subsys"] == 2].copy()
df_us["us_sipm_idx"] = df_us["global_sipm_idx"] - 224

fig, ax = plt.subplots(figsize=(15, 6))
df_us_left = df_us[df_us["side"] == "left"]
df_us_right = df_us[df_us["side"] == "right"]

ax.errorbar(df_us_left["us_sipm_idx"], df_us_left["d_sipm"], yerr=df_us_left["d_sipm_err"],
            fmt="+", color="red", markersize=6, capsize=0, label="left")
ax.errorbar(df_us_right["us_sipm_idx"], df_us_right["d_sipm"], yerr=df_us_right["d_sipm_err"],
            fmt="+", color="blue", markersize=6, capsize=0, label="right")

us_boundaries = [0, 160, 320, 480, 640, 800]
us_dividers = [80, 240, 400, 560, 720]
for b in us_boundaries:
    ax.axvline(b, color="lime", linewidth=2.0, linestyle="-")
for d in us_dividers:
    ax.axvline(d, color="magenta", linewidth=1.5, linestyle="--")

us_gel_pos = [(120, "G"), (200, "G"), (440, "G"), (520, "G"), (680, "G")]
for x_pos, text in us_gel_pos:
    ax.text(x_pos, 14.5, text, fontweight="bold", fontsize=13, ha="center", va="center")

ax.set_xlim(-10, 810)
ax.set_ylim(6, 16)
ax.set_xlabel("SiPM number", fontsize=14)
ax.set_ylabel(r"DS alignment parameter, $d_{\mathrm{SiPM}}$ [ns]", fontsize=14)
ax.set_title(f"DS alignment parameter, {{\mathrm{{SiPM}}}}$ as a function of SiPM number (Run {plot_run})", fontsize=15)
ax.grid(True, which="both", linestyle=":", alpha=0.5)
ax.legend(loc="upper right", frameon=True, framealpha=0.9, facecolor="white", edgecolor="black", fontsize=13)

plt.tight_layout()
plt.show()


---
## 6. Time-Walk Corrected 2D Distribution ($dt_{\mathrm{SiPM}}^{\mathrm{TW,ToF}}$ vs $\mathrm{QDC}_{\mathrm{SiPM}}$)

Visualizes the 2D distribution after subtracting the time-walk curve:
$$\Delta t^{\mathrm{TW}}(\mathrm{QDC}) = \frac{\alpha}{\beta \cdot \mathrm{QDC} - \mathrm{QDC}_0} + \gamma \cdot \mathrm{QDC}$$
This flattens the distribution across all QDC, directly illustrating the fine alignment parameter $d_{\mathrm{SiPM}}$ as the plateau level relative to zero (matching the dissertation demonstration plot).


In [ ]:
# Select channel and run to display
sample_run = 5408
target_ch = "24005_4"  # US Plane 5, Bar 5, SiPM 4 (Left side)
ref_type = "DS"

# Load channel data via loader utility
ch_data = load_channel_tw_root(sample_run, target_ch, ref_type=ref_type)
tw_p = ch_data["params"]
d_sipm = tw_p["alignment_d"]

# Plot 2D Time-Walk corrected distribution
fig, ax = plt.subplots(figsize=(10, 6))

masked_2d = np.ma.masked_where(ch_data["corr_2d"] <= 0, ch_data["corr_2d"])
mesh = ax.pcolormesh(ch_data["x_edges"], ch_data["y_edges"], masked_2d,
                     cmap="viridis",
                     norm=mcolors.LogNorm(vmin=1.0, vmax=max(10, np.nanmax(ch_data["corr_2d"]))),
                     shading="auto")
cbar = plt.colorbar(mesh, ax=ax, pad=0.02)
cbar.set_label("Counts (log scale)", fontsize=11)

ax.axhline(d_sipm, color="red", linestyle="--", linewidth=2.0)
ax.axhline(0.0, color="black", linestyle=":", linewidth=1.2, alpha=0.7)

ax.annotate("", xy=(165, 0.0), xytext=(165, d_sipm),
            arrowprops=dict(arrowstyle="<->", color="red", lw=2.0))
ax.text(145, 0.5 * d_sipm, rf"{{\mathrm{{SiPM}}}} = {d_sipm:.2f}$ ns",
        color="red", fontsize=11, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="red", alpha=0.9))

ax.set_xlabel(r"$\mathrm{QDC}_{\mathrm{SiPM}}$ [a.u.]", fontsize=12)
ax.set_ylabel(r"$\mathrm{dt}_{\mathrm{SiPM}}^{\mathrm{TW,ToF}}$ [ns]", fontsize=12)
ax.set_title(f"US plane 5, bar 5, SiPM 4 (left side) — $\mathrm{{dt}}_{{\mathrm{{SiPM}}}}^{{\mathrm{{TW,ToF}}}}$ vs $\mathrm{{QDC}}_{{\mathrm{{SiPM}}}}$ (Run {sample_run})", fontsize=12)
ax.set_xlim(0, 200)
ax.set_ylim(-2, 12)
ax.grid(True, linestyle=":", alpha=0.4)

plt.tight_layout()
plt.show()


---
## 7. Calibration Policy: Single Category Set vs. Run-by-Run Sets?

### Quantitative Assessment
To answer whether a single consolidated calibration set (such as `catII_timing_calibration.json`) can be used for all runs within a category or if run-dependent sets are required, we evaluate:
1. **Channel-Level Parameter Drift Over Runs:** Standard deviation of $d_{\mathrm{SiPM}}$ and $t_0$ across runs for each individual channel.
2. **Comparison with Intrinsic Time Resolution:** SiPM time resolution after TW correction is $\sim 250\text{--}350\,\mathrm{ps}$.
3. **Residual Distribution:** Channel-by-channel difference between individual run constants and the consolidated Category II calibration set.


In [ ]:
# Load Category II consolidated calibration
cat_file = os.path.join(fits_dir, "catII_timing_calibration.json")
has_cat_file = os.path.exists(cat_file)
if has_cat_file:
    with open(cat_file, "r") as f:
        cat_data = json.load(f)
    cat_al = cat_data.get("alignment", {})

# Compute per-channel drift (std across runs)
drift_records = []
for (ref, ch), group in df_channels.groupby(["ref", "channel"]):
    if len(group) < 5:
        continue
    d_std = group["d_sipm"].std()
    d_mean = group["d_sipm"].mean()
    t0_std = group["t0"].std()
    diff_to_cat = np.nan
    if has_cat_file and ref in cat_al and ch in cat_al[ref]:
        diff_to_cat = d_mean - cat_al[ref][ch][0]
    drift_records.append({
        "ref": ref,
        "channel": ch,
        "subsys": group["subsys"].iloc[0],
        "system": group["system"].iloc[0],
        "d_std": d_std,
        "t0_std": t0_std,
        "diff_to_cat": diff_to_cat,
    })

df_drift = pd.DataFrame(drift_records)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Run-to-run standard deviation of d_SiPM per channel
for ref, color in zip(["DS", "SF"], ["steelblue", "crimson"]):
    sub = df_drift[df_drift["ref"] == ref]["d_std"] * 1000.0  # in ps
    med_ps = sub.median()
    ax1.hist(sub, bins=40, range=(0, 400), alpha=0.6, color=color, edgecolor="black", linewidth=0.5,
             label=f"{ref} Ref (Median drift = {med_ps:.1f} ps)")
    ax1.axvline(med_ps, color=color, linestyle="--", linewidth=1.5)

ax1.axvline(260.0, color="black", linestyle=":", linewidth=2, label="SiPM Intrinsic Res ($\sim 260$ ps)")
ax1.set_xlabel(r"Channel-wise Run-to-Run Drift $\sigma_{\mathrm{run}}(d_{\mathrm{SiPM}})$ [ps]", fontsize=12)
ax1.set_ylabel("Number of Channels", fontsize=12)
ax1.set_title("Temporal Stability: Per-Channel Drift Across Runs")
ax1.grid(True, alpha=0.3)
ax1.legend(frameon=True, fontsize=10)

# Right plot: Difference between Run Average and Consolidated Category Set
if has_cat_file:
    for ref, color in zip(["DS", "SF"], ["steelblue", "crimson"]):
        sub = df_drift[df_drift["ref"] == ref]["diff_to_cat"].dropna() * 1000.0
        med_diff = sub.median()
        std_diff = sub.std()
        ax2.hist(sub, bins=40, range=(-300, 300), alpha=0.6, color=color, edgecolor="black", linewidth=0.5,
                 label=f"{ref} (Mean diff = {med_diff:.1f} ps, $\sigma$={std_diff:.1f} ps)")
    ax2.set_xlabel(r"{\mathrm{SiPM}}(\mathrm{Run\,Avg}) - d_{\mathrm{SiPM}}(\mathrm{Cat\,II})$ [ps]", fontsize=12)
    ax2.set_ylabel("Number of Channels", fontsize=12)
    ax2.set_title("Compatibility with Single Consolidated Category II Set")
    ax2.grid(True, alpha=0.3)
    ax2.legend(frameon=True, fontsize=10)

plt.tight_layout()
plt.show()

print("="*70)
print("CALIBRATION STABILITY SUMMARY:")
for ref in ["DS", "SF"]:
    sub = df_drift[df_drift["ref"] == ref]
    print(f"[{ref} Reference]")
    print(f"  -> Median channel drift across runs: {sub['d_std'].median()*1000.0:.1f} ps")
    print(f"  -> 90th percentile channel drift   : {sub['d_std'].quantile(0.90)*1000.0:.1f} ps")
print("="*70)


---
## 8. Fine Time Alignment ($d_{\mathrm{SiPM}}$) Extraction and $\pm 1\sigma$ Uncertainty Band

This plot illustrates how $d_{\mathrm{SiPM}}$ is extracted for a single SiPM channel (following Thesis Section 4.1.5 & Eq. 4.1.12):
1. **Left Panel (2D Distribution):** Shows $\mathrm{dt}_{\mathrm{SiPM}}^{\mathrm{TW,ToF}} = \Delta t - \Delta t^{\mathrm{TW}}(\mathrm{QDC})$ vs $\mathrm{QDC}$. The time-walk shape is removed, but the alignment offset $d_{\mathrm{SiPM}}$ remains. The red dashed line marks $d_{\mathrm{SiPM}}$, and the shaded red band shows the true $\pm 1\sigma$ distribution dispersion ($\sigma_{\mathrm{dist}} \approx 523\text{ ps}$) around the plateau.
2. **Right Panel (1D Projection):** The corresponding 1D projection of the plateau ($\{dt}_{\mathrm{SiPM}}^{\mathrm{TW,ToF}}$), highlighting $d_{\mathrm{SiPM}}$ and the $\pm 1\sigma$ truncated region from which $d_{\mathrm{SiPM}}$ and its uncertainty are extracted.


In [ ]:
# Load channel data via loader utility
data = load_channel_tw_root(sample_run=5408, target_ch="24005_4", ref_type="DS", y_min=6.0, y_max=14.0)
tw_params = data["params"]
d_sipm = tw_params["alignment_d"]
d_sipm_err = tw_params.get("alignment_d_err", 0.001)
sigma_dist = data["sigma_dist"]  # True 1-sigma distribution dispersion (~523 ps)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15.5, 6.2), gridspec_kw={"width_ratios": [2.3, 1.2]})

# 1. Left: 2D corrected distribution with true ±1 sigma (StdDev) band
masked_2d = np.ma.masked_where(data["corr_2d"] <= 0, data["corr_2d"])
mesh = ax1.pcolormesh(data["x_edges"], data["y_edges"], masked_2d, cmap="viridis",
                      norm=mcolors.LogNorm(vmin=1.0, vmax=max(10, np.nanmax(data["corr_2d"]))),
                      shading="auto")
cbar = plt.colorbar(mesh, ax=ax1, pad=0.02)
cbar.set_label("Counts (log scale)", fontsize=11)

ax1.axhline(d_sipm, color="red", linestyle="--", linewidth=2.0, label=rf"{{\mathrm{{SiPM}}}} = {d_sipm:.2f}\pm {d_sipm_err*1000:.1f}\\,\mathrm{{ps}}$")
ax1.axhspan(d_sipm - sigma_dist, d_sipm + sigma_dist, color="red", alpha=0.18, label=rf"$\pm 1\sigma$ Truncation Window ($\sigma_{{\mathrm{{dist}}}} = {sigma_dist*1000:.0f}\\,\mathrm{{ps}}$)")
ax1.axhline(d_sipm - sigma_dist, color="darkred", linestyle=":", linewidth=1.2)
ax1.axhline(d_sipm + sigma_dist, color="darkred", linestyle=":", linewidth=1.2)
ax1.axhline(0, color="black", linestyle=":", linewidth=1.0, alpha=0.7)

ax1.set_xlabel(r"$\mathrm{QDC}_{\mathrm{SiPM}}$ [a.u.]", fontsize=12)
ax1.set_ylabel(r"$\mathrm{dt}_{\mathrm{SiPM}}^{\mathrm{TW,ToF}}$ [ns]", fontsize=12)
ax1.set_title(f"Channel {data['channel']} (TW Corrected, Unaligned) with $\pm 1\sigma$ Truncation Window", fontsize=12.5)
ax1.set_xlim(0, 200)
ax1.set_ylim(6.0, 14.0)
ax1.grid(True, linestyle=":", alpha=0.4)
ax1.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.9, fontsize=10)

# 2. Right: 1D projection with shaded ±1 sigma truncated region
res_x = data["res_x"]
res_y = data["res_y"]
ax2.step(res_y, res_x, where="mid", color="steelblue", linewidth=1.5, label="1D Residual Projection")
ax2.fill_betweenx(res_x, 0, res_y, step="mid", color="steelblue", alpha=0.25)

# Highlight +/- 1 sigma band on 1D
mask_1sig = (res_x >= (d_sipm - sigma_dist)) & (res_x <= (d_sipm + sigma_dist))
ax2.fill_betweenx(res_x[mask_1sig], 0, res_y[mask_1sig], step="mid", color="red", alpha=0.35, label=rf"$\pm 1\sigma$ Truncated Region ({sigma_dist*1000:.0f} ps)")
ax2.axhline(d_sipm, color="red", linestyle="--", linewidth=2.0)
ax2.axhline(d_sipm - sigma_dist, color="darkred", linestyle=":", linewidth=1.2)
ax2.axhline(d_sipm + sigma_dist, color="darkred", linestyle=":", linewidth=1.2)

ax2.set_xlabel("Counts", fontsize=12)
ax2.set_ylabel(r"$\mathrm{dt}_{\mathrm{SiPM}}^{\mathrm{TW,ToF}}$ [ns]", fontsize=12)
ax2.set_title("1D Residual & Truncated Mean Region", fontsize=12.5)
ax2.set_ylim(6.0, 14.0)
ax2.grid(True, linestyle=":", alpha=0.4)
ax2.legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.9, fontsize=9.5)

plt.tight_layout()
plt.show()


---
## 8. Iteration 1 vs. Iteration 2 Comparative Analysis

Compares parameter convergence and timing resolution improvements between:
- **Iteration 1:** Standard extraction without prior {\mathrm{scint}}$ time-walk correction.
- **Iteration 2 ():** Extracted using effective velocities {\mathrm{scint}}^{\mathrm{TW}}$.


In [ ]:
# Merge channel calibrations on (run, channel, ref)
df_comp = pd.merge(
    df_channels_iter1[["run", "channel", "ref", "subsys", "system", "d_sipm", "alpha", "beta", "sigma_sys", "rms_residual"]],
    df_channels_tw[["run", "channel", "ref", "d_sipm", "alpha", "beta", "sigma_sys", "rms_residual"]],
    on=["run", "channel", "ref"],
    suffixes=("_iter1", "_iter2")
)

df_comp["delta_d"] = df_comp["d_sipm_iter2"] - df_comp["d_sipm_iter1"]
df_comp["delta_alpha"] = df_comp["alpha_iter2"] - df_comp["alpha_iter1"]
df_comp["delta_sigma_sys"] = df_comp["sigma_sys_iter2"] - df_comp["sigma_sys_iter1"]

print(f"Total matched channel calibration pairs: {len(df_comp):,}")

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 11))

# 1. d_sipm comparison
for sys, col in [("US", "#1f77b4"), ("DS", "#ff7f0e"), ("Veto", "#2ca02c")]:
    s_df = df_comp[(df_comp["system"] == sys) & (df_comp["ref"] == "DS") if sys != "Veto" else (df_comp["system"] == sys)]
    ax1.scatter(s_df["d_sipm_iter1"], s_df["d_sipm_iter2"], alpha=0.35, s=12, color=col, label=f"{sys} ({len(s_df):,})")
ax1.plot([-5, 5], [-5, 5], color="red", linestyle="--", label="Identity")
ax1.set_xlim(-4, 4)
ax1.set_ylim(-4, 4)
ax1.set_xlabel(r"{\mathrm{SiPM}}$ Iteration 1 [ns]", fontsize=11)
ax1.set_ylabel(r"{\mathrm{SiPM}}$ Iteration 2 () [ns]", fontsize=11)
ax1.set_title("Alignment Parameter {\mathrm{SiPM}}$ Convergence", fontweight="bold")
ax1.grid(True, linestyle=":", alpha=0.5)
ax1.legend(loc="upper left", fontsize=9.5)

# 2. Delta d distribution
for sys, col in [("US", "#1f77b4"), ("DS", "#ff7f0e"), ("Veto", "#2ca02c")]:
    s_vals = df_comp[(df_comp["system"] == sys)]["delta_d"].dropna()
    ax2.hist(s_vals, bins=np.linspace(-0.5, 0.5, 70), density=True, histtype="step", linewidth=2.0, color=col,
             label=f"{sys}: Med = {np.median(s_vals):+.3f} ns (RMS = {np.std(s_vals):.3f})")
ax2.axvline(0, color="black", linestyle=":")
ax2.set_xlabel(r"$\Delta d = d_{\mathrm{iter2}} - d_{\mathrm{iter1}}$ [ns]", fontsize=11)
ax2.set_ylabel("Density", fontsize=11)
ax2.set_title("Fine Time Alignment Stability ($\Delta d$)", fontweight="bold")
ax2.grid(True, linestyle=":", alpha=0.5)
ax2.legend(loc="upper right", fontsize=9.5)

# 3. Alpha correlation
for sys, col in [("US", "#1f77b4"), ("DS", "#ff7f0e")]:
    s_df = df_comp[(df_comp["system"] == sys) & (df_comp["alpha_iter1"].between(0, 150)) & (df_comp["alpha_iter2"].between(0, 150))]
    ax3.scatter(s_df["alpha_iter1"], s_df["alpha_iter2"], alpha=0.35, s=12, color=col, label=f"{sys} (Med={s_df['alpha_iter2'].median():.1f})")
ax3.plot([0, 150], [0, 150], color="red", linestyle="--")
ax3.set_xlabel(r"$\alpha$ Iteration 1", fontsize=11)
ax3.set_ylabel(r"$\alpha$ Iteration 2 ()", fontsize=11)
ax3.set_title("Time-Walk Amplitude $\alpha$ Comparison", fontweight="bold")
ax3.grid(True, linestyle=":", alpha=0.5)
ax3.legend(loc="upper left", fontsize=9.5)

# 4. Systematic Floor Comparison (sigma_sys)
for sys, col in [("US", "#1f77b4"), ("DS", "#ff7f0e")]:
    s_vals = (df_comp[df_comp["system"] == sys]["delta_sigma_sys"].dropna()) * 1000 # in ps
    ax4.hist(s_vals, bins=np.linspace(-150, 150, 60), density=True, histtype="step", linewidth=2.0, color=col,
             label=f"{sys}: Med Shift = {np.median(s_vals):+.1f} ps (RMS = {np.std(s_vals):.1f})")
ax4.axvline(0, color="black", linestyle=":")
ax4.set_xlabel(r"$\Delta \sigma_{\mathrm{sys}} = \sigma_{\mathrm{sys}}^{\mathrm{iter2}} - \sigma_{\mathrm{sys}}^{\mathrm{iter1}}$ [ps]", fontsize=11)
ax4.set_ylabel("Density", fontsize=11)
ax4.set_title("Systematic Error Floor Variation ($\Delta \sigma_{\mathrm{sys}}$)", fontweight="bold")
ax4.grid(True, linestyle=":", alpha=0.5)
ax4.legend(loc="upper right", fontsize=9.5)

plt.tight_layout()
plt.show()

print("=== Summary of Iteration 1 vs. Iteration 2 Calibration Convergence ===")
for sys in ["US", "DS", "Veto"]:
    sub = df_comp[df_comp["system"] == sys]
    dd = sub["delta_d"].dropna()
    dsig = (sub["delta_sigma_sys"].dropna()) * 1000
    print(f"{sys:5s} | Matched: {len(sub):5d} | Med Delta d: {np.median(dd):+.3f} ns (RMS: {np.std(dd):.3f} ns) | "
          f"Med Delta sigma_sys: {np.median(dsig):+.1f} ps")


---
## 9. Run-by-Run Dependencies: Intra-Year vs. Inter-Year Parameter Stability

Direct quantitative evaluation of parameter stability:
1. **Intra-Year Stability (Within Same Year):** Channel-level dispersion $\sigma(p_i)$ across runs within the same data-taking period (e.g. Cat II 2022).
2. **Inter-Year Stability (Between Different Years):** Systematic channel shift $\Delta p = p^{2023} - p^{2022}$ and ^{2025} - p^{2023}$ across LHC operational winter shutdowns.


In [ ]:
# 1. Intra-Year Dispersion per Channel within 2022 (Cat II)
df_cat2 = df_channels_tw[(df_channels_tw["period"] == "Cat II (2022)") & (df_channels_tw["ref"] == "DS")].copy()
ch_intra_std = df_cat2.groupby("channel")[["d_sipm", "alpha", "beta", "qdc0", "sigma_sys"]].std()

# 2. Inter-Year Shifts (2023 vs 2022 and 2025 vs 2023)
# Group by channel and period to get period-median per channel
ch_period_meds = df_channels_tw[df_channels_tw["ref"] == "DS"].groupby(["channel", "period"])[["d_sipm", "alpha", "beta", "sigma_sys"]].median().unstack()

diff_23_22_d = (ch_period_meds["d_sipm"]["Cat III (2023)"] - ch_period_meds["d_sipm"]["Cat II (2022)"]).dropna()
diff_25_23_d = (ch_period_meds["d_sipm"]["Cat III (2025)"] - ch_period_meds["d_sipm"]["Cat III (2023)"]).dropna()

diff_23_22_a = (ch_period_meds["alpha"]["Cat III (2023)"] - ch_period_meds["alpha"]["Cat II (2022)"]).dropna()
diff_25_23_a = (ch_period_meds["alpha"]["Cat III (2025)"] - ch_period_meds["alpha"]["Cat III (2023)"]).dropna()

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Panel 1: Intra-year channel RMS of d_sipm (within 2022)
ax1.hist(ch_intra_std["d_sipm"].dropna() * 1000, bins=np.linspace(0, 150, 50), color="#1f77b4", alpha=0.75, edgecolor="black")
ax1.axvline(np.median(ch_intra_std["d_sipm"].dropna() * 1000), color="red", linestyle="--", linewidth=2,
            label=f"Median Intra-Year $\sigma(d)$: {np.median(ch_intra_std['d_sipm'].dropna() * 1000):.1f} ps")
ax1.set_xlabel(r"Intra-Year Channel RMS $\sigma(d_{\mathrm{SiPM}})$ [ps]", fontsize=11)
ax1.set_ylabel("Channels Count", fontsize=11)
ax1.set_title("Intra-Year Stability: Run-to-Run {\mathrm{SiPM}}$ Variation (2022)", fontweight="bold")
ax1.grid(True, linestyle=":", alpha=0.5)
ax1.legend(loc="upper right", fontsize=10)

# Panel 2: Inter-year systematic shifts of d_sipm
ax2.hist(diff_23_22_d, bins=np.linspace(-2.0, 2.0, 60), histtype="step", linewidth=2.2, color="#ff7f0e",
         label=f"2023 - 2022: Med = {np.median(diff_23_22_d):+.3f} ns (RMS = {np.std(diff_23_22_d):.3f})")
ax2.hist(diff_25_23_d, bins=np.linspace(-2.0, 2.0, 60), histtype="step", linewidth=2.2, color="#2ca02c",
         label=f"2025 - 2023: Med = {np.median(diff_25_23_d):+.3f} ns (RMS = {np.std(diff_25_23_d):.3f})")
ax2.axvline(0, color="black", linestyle=":", linewidth=1.2)
ax2.set_xlabel(r"Inter-Year Shift $\Delta d_{\mathrm{SiPM}}$ [ns]", fontsize=11)
ax2.set_ylabel("Channels Count", fontsize=11)
ax2.set_title("Inter-Year Shift: {\mathrm{SiPM}}$ Year-to-Year Shifts", fontweight="bold")
ax2.grid(True, linestyle=":", alpha=0.5)
ax2.legend(loc="upper right", fontsize=9.5)

# Panel 3: Intra-year channel RMS of alpha (within 2022)
ax3.hist(ch_intra_std["alpha"].dropna(), bins=np.linspace(0, 10, 50), color="#9467bd", alpha=0.75, edgecolor="black")
ax3.axvline(np.median(ch_intra_std["alpha"].dropna()), color="red", linestyle="--", linewidth=2,
            label=f"Median Intra-Year $\sigma(\alpha)$: {np.median(ch_intra_std['alpha'].dropna()):.2f}")
ax3.set_xlabel(r"Intra-Year Channel RMS $\sigma(\alpha)$", fontsize=11)
ax3.set_ylabel("Channels Count", fontsize=11)
ax3.set_title("Intra-Year Stability: Run-to-Run $\alpha$ Variation (2022)", fontweight="bold")
ax3.grid(True, linestyle=":", alpha=0.5)
ax3.legend(loc="upper right", fontsize=10)

# Panel 4: Inter-year systematic shifts of alpha
ax4.hist(diff_23_22_a, bins=np.linspace(-15, 15, 60), histtype="step", linewidth=2.2, color="#ff7f0e",
         label=f"2023 - 2022: Med = {np.median(diff_23_22_a):+.2f} (RMS = {np.std(diff_23_22_a):.2f})")
ax4.hist(diff_25_23_a, bins=np.linspace(-15, 15, 60), histtype="step", linewidth=2.2, color="#2ca02c",
         label=f"2025 - 2023: Med = {np.median(diff_25_23_a):+.2f} (RMS = {np.std(diff_25_23_a):.2f})")
ax4.axvline(0, color="black", linestyle=":", linewidth=1.2)
ax4.set_xlabel(r"Inter-Year Shift $\Delta \alpha$", fontsize=11)
ax4.set_ylabel("Channels Count", fontsize=11)
ax4.set_title("Inter-Year Shift: Shape Parameter $\alpha$ Continuity", fontweight="bold")
ax4.grid(True, linestyle=":", alpha=0.5)
ax4.legend(loc="upper right", fontsize=9.5)

plt.tight_layout()
plt.show()

print("=== Summary: Intra-Year vs. Inter-Year Parameter Variations ===")
med_d_std = np.median(ch_intra_std["d_sipm"].dropna() * 1000)
med_a_std = np.median(ch_intra_std["alpha"].dropna())
med_s_std = np.median(ch_intra_std["sigma_sys"].dropna() * 1000)
shift_d_23 = np.median(diff_23_22_d)
shift_d_25 = np.median(diff_25_23_d)
shift_a_23 = np.median(diff_23_22_a)
shift_a_25 = np.median(diff_25_23_a)

print("=== Summary: Intra-Year vs. Inter-Year Parameter Variations ===")
print(f"1. Intra-Year (Run-by-Run within 2022):")
print(f"   - d_SiPM RMS per channel : {med_d_std:.1f} ps (negligible clock drift)")
print(f"   - alpha RMS per channel  : {med_a_std:.2f} (shape is static)")
print(f"   - sigma_sys RMS          : {med_s_std:.1f} ps")
print(f"2. Inter-Year (Between Years across YETS):")
print(f"   - d_SiPM Shift 2023-2022 : {shift_d_23:+.3f} ns (bulk clock re-alignment)")
print(f"   - d_SiPM Shift 2025-2023 : {shift_d_25:+.3f} ns")
print(f"   - alpha Shift 2023-2022  : {shift_a_23:+.2f} (near zero)")
print(f"   - alpha Shift 2025-2023  : {shift_a_25:+.2f} (near zero)")


---
## 10. Intra-Category Run-by-Run Parameter Variations (Cat II 2022, Cat III 2023, Cat III 2025)

Visualizes the evolution of the primary time-walk parameters ({\mathrm{SiPM}}$, $lpha$, $eta$, $, and $\sigma_{\mathrm{sys}}$) across runs within each period:
- **Color Coding:** Distinct discrete colormap by **SiPM channel number (0 to 15)**.
- **Trend Overlay:** Run-by-run aggregate median (solid black diamonds) and global period median (dashed line).


In [ ]:
# Define periods and parameters to display
target_periods = ["Cat II (2022)", "Cat III (2023)", "Cat III (2025)"]
params_to_plot = [
    ("d_sipm",    r"{\mathrm{SiPM}}$ [ns]",             "Fine Time Alignment ({\mathrm{SiPM}}$)",   (5.0, 13.0)),
    ("alpha",     r"$\alpha$",                            "Curvature Scale ($\alpha$)",                  (-80.0, 0.0)),
    ("beta",      r"$\beta$",                             "Scale Factor ($\beta$)",                     (0.5, 4.0)),
    ("qdc0",      r"$ [QDC]",                         "Threshold Offset ($)",                     (-15.0, -3.0)),
    ("sigma_sys", r"$\sigma_{\mathrm{sys}}$ [ns]",          "Systematic Uncertainty Floor ($\sigma_{\mathrm{sys}}$)", (0.04, 0.15)),
]

cmap_sipm = plt.colormaps["turbo"].resampled(16)
norm_sipm = mcolors.Normalize(vmin=-0.5, vmax=15.5)

# Filter to DS reference channels in target periods
df_plot_tw = df_channels_tw[
    (df_channels_tw["period"].isin(target_periods)) &
    (df_channels_tw["ref"] == "DS")
].copy()

for p_name in target_periods:
    p_data = df_plot_tw[df_plot_tw["period"] == p_name]
    if p_data.empty:
        continue

    fig, axes = plt.subplots(len(params_to_plot), 1, figsize=(15, 15), sharex=True, layout="constrained")
    runs_in_p = sorted(p_data["run"].unique())

    for ax, (param_col, ylabel, title, ylim) in zip(axes, params_to_plot):
        # Clean valid values for this parameter
        valid = p_data[p_data[param_col].notna() & np.isfinite(p_data[param_col])]

        sc = ax.scatter(
            valid["run"], valid[param_col],
            c=valid["sipm"], cmap=cmap_sipm, norm=norm_sipm,
            alpha=0.35, s=14, edgecolors="none", rasterized=True, zorder=2
        )

        # Aggregate run median
        run_meds = valid.groupby("run")[param_col].median()
        ax.plot(
            run_meds.index, run_meds.values,
            color="black", marker="D", markersize=3.5, linewidth=1.2, linestyle="-",
            alpha=0.9, label=f"Run Median (N={len(run_meds)})", zorder=4
        )

        # Global period median
        glob_med = float(np.median(valid[param_col].dropna()))
        ax.axhline(glob_med, color="dimgray", linestyle="--", linewidth=1.2, alpha=0.85,
                   label=f"Period Median: {glob_med:.3f}", zorder=3)

        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(title, fontsize=12, fontweight="bold", pad=5)
        ax.set_ylim(ylim)
        ax.grid(True, linestyle=":", alpha=0.5)
        ax.legend(loc="upper right", framealpha=0.92, fontsize=9.5)

    axes[-1].set_xlabel("Run Number", fontsize=12)
    cbar = fig.colorbar(sc, ax=axes, orientation="vertical", fraction=0.022, pad=0.02, ticks=range(16))
    cbar.set_label("SiPM Channel ID (0–7: Left, 8–15: Right)", fontsize=11)
    cbar.ax.tick_params(labelsize=9.5)
    fig.suptitle(f"SND@LHC Time-Walk Parameters: Intra-Category Stability — {p_name}", fontsize=14, fontweight="bold", y=1.002)
    plt.show()


---
## 11. Cross-Category Parameter Variations (Cat II 2022 vs. Cat III 2023 vs. Cat III 2025)

Contrasts parameter distributions across the three operational periods with categorical x-axis:
- **{\mathrm{SiPM}}$:** Shows the bulk LHC clock realignment shift between operational years.
- **$lpha, eta, Q_0$:** Demonstrates the invariance of the intrinsic time-walk pulse shaping across categories.
- **$\sigma_{\mathrm{sys}}$:** Confirms timing resolution convergence across all eras.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5.5))

colors_period = ["#1f77b4", "#ff7f0e", "#d62728"]
period_labels = ["Cat II (2022)", "Cat III (2023)", "Cat III (2025)"]

for ax, (param_col, ylabel, title, ylim) in zip(axes, params_to_plot):
    box_data = []
    for p in target_periods:
        vals = df_plot_tw[df_plot_tw["period"] == p][param_col].dropna().values
        # Filter extreme unconstrained outliers for clean boxplot representation
        vals_clean = vals[(vals >= ylim[0]) & (vals <= ylim[1])]
        box_data.append(vals_clean)

    bp = ax.boxplot(
        box_data, labels=period_labels, patch_artist=True, showmeans=True,
        meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 5},
        flierprops={"marker": ".", "markersize": 2, "alpha": 0.2}
    )

    for patch, col in zip(bp["boxes"], colors_period):
        patch.set_facecolor(col)
        patch.set_alpha(0.65)

    # Add median labels
    for i, vals_clean in enumerate(box_data):
        med = np.median(vals_clean)
        ax.text(i + 1, ylim[0] + 0.05 * (ylim[1] - ylim[0]), f"{med:.2f}",
                ha="center", va="bottom", fontsize=9, fontweight="bold", color="black",
                bbox=dict(boxstyle="round,pad=0.15", facecolor="white", alpha=0.8, edgecolor="gray"))

    ax.set_title(title.split("(")[0].strip(), fontsize=11, fontweight="bold", pad=8)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_ylim(ylim)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.tick_params(labelsize=10)

fig.suptitle("SND@LHC Time-Walk Parameters: Cross-Category Comparison", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Tabular summary of cross-category metrics:
# Showing Median +/- SE(Median) [true uncertainty of the measurement]
# and [IQR / 1.349] as the detector channel spread
rows_cross = []
for p in target_periods:
    sub = df_plot_tw[df_plot_tw["period"] == p]
    
    def get_med_and_unc(series, scale=1.0):
        vals = series.dropna().values * scale
        if len(vals) == 0:
            return "N/A"
        med = np.median(vals)
        iqr = np.percentile(vals, 75) - np.percentile(vals, 25)
        se_med = 1.253 * (iqr / 1.349) / np.sqrt(len(vals))
        return f"{med:.2f} +/- {se_med:.3f}"

    rows_cross.append({
        "Period": p,
        "Runs": sub["run"].nunique(),
        "d_SiPM [ns]": get_med_and_unc(sub["d_sipm"]),
        "alpha": get_med_and_unc(sub["alpha"]),
        "beta": get_med_and_unc(sub["beta"]),
        "Q0 [QDC]": get_med_and_unc(sub["qdc0"]),
        "sigma_sys [ps]": get_med_and_unc(sub["sigma_sys"], scale=1000.0)
    })
df_summary_cross = pd.DataFrame(rows_cross)
print("=== Cross-Category Parameters: Robust Median +/- Measurement Uncertainty (SE) ===")
print(df_summary_cross.to_string(index=False))


---
## 12. Quality Selection & Unified Time-Walk Calibration Extraction

### 12.1 Good-Fit Selection Criteria
Removes unconstrained fit excursions and boundary-hit MINUIT solutions:
- $\chi^2_{\mathrm{tot}}/\mathrm{ndf} \leq 2.0$
- $\sigma_{\mathrm{sys}} \leq 0.15~\mathrm{ns}$
- hBc80.0 \leq lpha \leq -5.0$
- bash.2 \leq eta \leq 4.0$
- hBc15.0 \leq Q_0 \leq -2.0$

### 12.2 Single Unified Parameter Set per SiPM Channel
- **Time-Invariant Shape Parameters:** $\hat{lpha}_i, \hat{eta}_i, \hat{Q}_{0,i}, \hat{\sigma}_{\mathrm{sys},i}$ pooled via median over all available runs.
- **Period-Specific Fine Alignment:** $\hat{d}_i^{(2022)}, \hat{d}_i^{(2023)}, \hat{d}_i^{(2025)}$ pooled per operational era to preserve the bulk LHC clock phase.


In [ ]:
# Define Good-Fit Mask on df_channels_tw (DS reference)
df_ds = df_channels_tw[df_channels_tw["ref"] == "DS"].copy()

good_fit_mask = (
    (df_ds["chi2_tot_ndf"].notna()) & (df_ds["chi2_tot_ndf"] <= 2.0) &
    (df_ds["sigma_sys"].notna()) & (df_ds["sigma_sys"] <= 0.15) & (df_ds["sigma_sys"] > 0.0) &
    (df_ds["alpha"].notna()) & (df_ds["alpha"] >= -80.0) & (df_ds["alpha"] <= -5.0) &
    (df_ds["beta"].notna()) & (df_ds["beta"] >= 0.2) & (df_ds["beta"] <= 4.0) &
    (df_ds["qdc0"].notna()) & (df_ds["qdc0"] >= -15.0) & (df_ds["qdc0"] <= -2.0)
)

df_good = df_ds[good_fit_mask].copy()
n_total = len(df_ds)
n_good = len(df_good)
print(f"Time-Walk Good-Fit Selection: {n_good:,} / {n_total:,} channel fits retained ({n_good / n_total * 100:.1f}%)")

# Extract single unified shape parameters per SiPM channel across all runs
unified_shape = df_good.groupby(["channel", "detID", "subsys", "system", "plane_label", "sipm", "side"]).agg(
    n_runs=("run", "count"),
    alpha_hat=("alpha", "median"),
    alpha_iqr=("alpha", lambda x: np.percentile(x, 75) - np.percentile(x, 25)),
    beta_hat=("beta", "median"),
    beta_iqr=("beta", lambda x: np.percentile(x, 75) - np.percentile(x, 25)),
    qdc0_hat=("qdc0", "median"),
    qdc0_iqr=("qdc0", lambda x: np.percentile(x, 75) - np.percentile(x, 25)),
    sigma_sys_hat=("sigma_sys", "median"),
).reset_index()

# Extract period-specific fine alignment d_sipm
d_by_period = df_good.groupby(["channel", "period"])["d_sipm"].median().unstack()
col_map = {
    "Cat II (2022)": "d_hat_catII_2022",
    "Cat III (2023)": "d_hat_catIII_2023",
    "Cat III (2025)": "d_hat_catIII_2025",
}
d_by_period.rename(columns=col_map, inplace=True)

df_unified_calib = pd.merge(unified_shape, d_by_period, on="channel", how="left")
print(f"Generated Unified Time-Walk Calibration Table for {len(df_unified_calib):,} SiPM channels.")
df_unified_calib.head(10)


---
## 13. Statistical Quantification of Parameter Time-Independence

Evaluates whether the variation of time-walk parameters is consistent with zero temporal drift across all SiPM channels:
1. **Linear Drift Regression:** Fits (r) = m_i \cdot r + c_i$ for each channel over run number $.
2. **Student's hBctest on Slope:**  = rac{m_i}{\mathrm{SE}(m_i)}$. The null hypothesis ( = 0$) is accepted if $|t_i| < 2.0$ ( > 0.05$).
3. **Coefficient of Variation ($):**  = rac{\mathrm{IQR}(p_i)}{1.349 \cdot |\mathrm{median}(p_i)|}$, evaluating relative dispersion.
4. **Global Detector Stability Fraction ({\mathrm{stable}}$):** Percentage of detector channels meeting $|t_i| < 2.0$.


In [ ]:
from scipy import stats

# Compute time-independence drift slope and t-statistic for each channel
drift_records = []
min_runs_for_drift = 8

for ch_key, ch_data in df_good.groupby("channel"):
    if len(ch_data) < min_runs_for_drift:
        continue
    r_vals = ch_data["run"].values
    ch_rec = {"channel": ch_key, "subsys": ch_data["system"].iloc[0], "n_runs": len(ch_data)}

    for p in ["alpha", "beta", "qdc0", "sigma_sys", "d_sipm"]:
        vals = ch_data[p].values
        med = np.median(vals)
        iqr = np.percentile(vals, 75) - np.percentile(vals, 25)
        norm_disp = iqr / (1.349 * abs(med)) if abs(med) > 1e-4 else np.nan

        # Linear regression vs run number
        res = stats.linregress(r_vals, vals)
        slope = res.slope
        slope_err = res.stderr if res.stderr is not None and res.stderr > 0 else 1e-9
        t_stat = slope / slope_err
        p_val = res.pvalue
        total_drift = slope * (r_vals.max() - r_vals.min())

        ch_rec[f"{p}_slope"] = slope
        ch_rec[f"{p}_t_stat"] = t_stat
        ch_rec[f"{p}_p_val"] = p_val
        ch_rec[f"{p}_cv"] = norm_disp
        ch_rec[f"{p}_drift"] = total_drift
        ch_rec[f"{p}_is_stable"] = (abs(t_stat) < 2.0)

    drift_records.append(ch_rec)

df_drift = pd.DataFrame(drift_records)
print(f"Evaluated time-independence for {len(df_drift):,} SiPM channels with >= {min_runs_for_drift} good runs.")

# Global Stability Summary Table
summary_drift = []
for p in ["alpha", "beta", "qdc0", "sigma_sys", "d_sipm"]:
    t_vals = df_drift[f"{p}_t_stat"].dropna()
    cv_vals = df_drift[f"{p}_cv"].dropna() * 100  # in %
    drift_vals = df_drift[f"{p}_drift"].dropna()
    f_stable = (df_drift[f"{p}_is_stable"].sum() / len(df_drift)) * 100
    summary_drift.append({
        "Parameter": p,
        "Stable Channels (|t| < 2)": f"{f_stable:.1f}%",
        "Median Relative CV [%]": f"{np.median(cv_vals):.2f}%",
        "Median |t-statistic|": f"{np.median(np.abs(t_vals)):.2f}",
        "Median Total Drift": f"{np.median(drift_vals):+.4f}",
        "RMS Total Drift": f"{np.std(drift_vals):.4f}"
    })
print("=== Time-Independence Summary Across All Evaluated SiPM Channels ===")
print(pd.DataFrame(summary_drift).to_string(index=False))

# Diagnostic Visualizations
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# 1. t-statistic distribution for alpha and beta
bins_t = np.linspace(-4, 4, 60)
ax1.hist(df_drift["alpha_t_stat"].dropna(), bins=bins_t, histtype="step", linewidth=2.0, color="#1f77b4",
         label=f"$\alpha$: Stable = {(df_drift['alpha_is_stable'].mean())*100:.1f}%")
ax1.hist(df_drift["beta_t_stat"].dropna(), bins=bins_t, histtype="step", linewidth=2.0, color="#ff7f0e",
         label=f"$\beta$: Stable = {(df_drift['beta_is_stable'].mean())*100:.1f}%")
ax1.axvspan(-2.0, 2.0, color="green", alpha=0.10, label="Null Hypothesis Retained (|t| < 2)")
ax1.axvline(-2.0, color="green", linestyle="--", linewidth=1.2)
ax1.axvline(2.0, color="green", linestyle="--", linewidth=1.2)
ax1.set_xlabel("Student's hBcstatistic on Linear Slope ( / \sigma_m$)", fontsize=11)
ax1.set_ylabel("Channels Count", fontsize=11)
ax1.set_title("Temporal Drift Significance Test (hBcdistribution)", fontweight="bold")
ax1.grid(True, linestyle=":", alpha=0.5)
ax1.legend(loc="upper right", fontsize=9.5)

# 2. Relative Dispersion CV (%) across all parameters
bins_cv = np.linspace(0, 20, 50)
ax2.hist(df_drift["alpha_cv"].dropna() * 100, bins=bins_cv, histtype="step", linewidth=2.0, color="#1f77b4",
         label=f"$\alpha$ CV: Med = {np.median(df_drift['alpha_cv'].dropna())*100:.1f}%")
ax2.hist(df_drift["beta_cv"].dropna() * 100, bins=bins_cv, histtype="step", linewidth=2.0, color="#ff7f0e",
         label=f"$\beta$ CV: Med = {np.median(df_drift['beta_cv'].dropna())*100:.1f}%")
ax2.hist(df_drift["qdc0_cv"].dropna() * 100, bins=bins_cv, histtype="step", linewidth=2.0, color="#d62728",
         label=f"$ CV: Med = {np.median(df_drift['qdc0_cv'].dropna())*100:.1f}%")
ax2.set_xlabel("Coefficient of Variation (Normalized Dispersion) [%]", fontsize=11)
ax2.set_ylabel("Channels Count", fontsize=11)
ax2.set_title("Relative Parameter Stability Across Runs ($)", fontweight="bold")
ax2.grid(True, linestyle=":", alpha=0.5)
ax2.legend(loc="upper right", fontsize=9.5)

# 3. Cleaned Trajectories of Specific SiPMs with Good-Fit Selection
selected_chs = ["24004_4", "21007_8", "20004_12", "23000_15"]
colors_sel = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd"]
for ch, col in zip(selected_chs, colors_sel):
    s_data = df_good[df_good["channel"] == ch].sort_values("run")
    if not s_data.empty:
        ax3.plot(s_data["run"], s_data["alpha"], marker="o", markersize=3.5, linewidth=1.1, color=col, alpha=0.85, label=ch)
ax3.set_ylabel(r"$lpha$ (Curvature)", fontsize=11)
ax3.set_xlabel("Run Number", fontsize=11)
ax3.set_title("Cleaned Good-Fit $\alpha$ Trajectories (Selected SiPMs)", fontweight="bold")
ax3.set_ylim(-45, -15)
ax3.grid(True, linestyle=":", alpha=0.5)
ax3.legend(loc="upper right", fontsize=9.5)

# 4. Cleaned Trajectories of beta for Specific SiPMs
for ch, col in zip(selected_chs, colors_sel):
    s_data = df_good[df_good["channel"] == ch].sort_values("run")
    if not s_data.empty:
        ax4.plot(s_data["run"], s_data["beta"], marker="s", markersize=3.5, linewidth=1.1, color=col, alpha=0.85, label=ch)
ax4.set_ylabel(r"$eta$ (Scale)", fontsize=11)
ax4.set_xlabel("Run Number", fontsize=11)
ax4.set_title("Cleaned Good-Fit $\beta$ Trajectories (Selected SiPMs)", fontweight="bold")
ax4.set_ylim(0.5, 2.5)
ax4.grid(True, linestyle=":", alpha=0.5)
ax4.legend(loc="upper right", fontsize=9.5)

plt.tight_layout()
plt.show()
